# 2916. 子数组不同元素数目的平方和 II

按右端点枚举。逻辑数组 d[l] 表示当前子数组 nums[l:r+1] 的不同元素数，用区间加、区间求和线段树维护。

[原文](https://leetcode.cn/problems/subarrays-distinct-element-sum-of-squares-ii/solution/2916xian-duan-shu-xian-bie-ji-zhao-bei-b-fj90/) · 整理于 2026-10-17。原代码保持原样；补充注释另放单元格。

## 平方差化为两次区间操作

x 上次出现在 prev，本轮只有左端点 `[prev+1,r]` 的不同计数加 1。若旧计数和为 S、区间长度为 m，平方和增量为 `2*S+m`。

In [ ]:
# last 保存处理当前位置之前的最后出现下标；没出现过则为 -1。
# ending：仅以当前 r 结尾的所有子数组贡献；answer：累计所有右端点贡献。
# 先 query 旧计数和，再 add；新位置 r 的单元素子数组已包含在更新中。
# tree[p] 是该区间已经更新的和；lazy[p] 是尚未传给孩子的每元素增量。
# 完全覆盖直接处理；进入孩子前 push；修改后把孩子之和合并回父节点。

In [ ]:
from typing import List


class LazySegmentTree:
    """区间加、区间求和；数组下标从 0 开始，区间均为闭区间。"""

    def __init__(self, nums: list[int]):
        self.n = len(nums)
        self.tree = [0] * (4 * self.n)
        self.lazy = [0] * (4 * self.n)
        self._build(1, 0, self.n - 1, nums)

    def _build(self, p, l, r, nums):
        if l == r:
            self.tree[p] = nums[l]
            return
        mid = (l + r) // 2
        self._build(p * 2, l, mid, nums)
        self._build(p * 2 + 1, mid + 1, r, nums)
        self.tree[p] = self.tree[p * 2] + self.tree[p * 2 + 1]

    def _apply(self, p, l, r, value):
        self.tree[p] += (r - l + 1) * value
        self.lazy[p] += value

    def _push(self, p, l, r):
        if self.lazy[p] != 0:
            mid = (l + r) // 2
            self._apply(p * 2, l, mid, self.lazy[p])
            self._apply(p * 2 + 1, mid + 1, r, self.lazy[p])
            self.lazy[p] = 0

    def add(self, left, right, value):
        self._add(1, 0, self.n - 1, left, right, value)

    def _add(self, p, l, r, ql, qr, value):
        if ql <= l and r <= qr:
            self._apply(p, l, r, value)
            return
        self._push(p, l, r)
        mid = (l + r) // 2
        if ql <= mid:
            self._add(p * 2, l, mid, ql, qr, value)
        if qr > mid:
            self._add(p * 2 + 1, mid + 1, r, ql, qr, value)
        self.tree[p] = self.tree[p * 2] + self.tree[p * 2 + 1]

    def query(self, left, right):
        return self._query(1, 0, self.n - 1, left, right)

    def _query(self, p, l, r, ql, qr):
        if ql <= l and r <= qr:
            return self.tree[p]
        self._push(p, l, r)
        mid = (l + r) // 2
        result = 0
        if ql <= mid:
            result += self._query(p * 2, l, mid, ql, qr)
        if qr > mid:
            result += self._query(p * 2 + 1, mid + 1, r, ql, qr)
        return result


class Solution:
    def sumCounts(self, nums: List[int]) -> int:
        mod = 10**9 + 7
        seg = LazySegmentTree([0] * len(nums))
        last = {}
        ending = 0  # 以上一个位置为右端点的所有子数组，其不同计数的平方和。
        answer = 0

        for r, x in enumerate(nums):
            prev = last.get(x, -1)
            left = prev + 1
            old_sum = seg.query(left, r)
            ending += 2 * old_sum + (r - left + 1)
            seg.add(left, r, 1)
            answer = (answer + ending) % mod
            last[x] = r

        return answer

In [ ]:
assert Solution().sumCounts([1]) == 1
assert Solution().sumCounts([1, 2, 1]) == 15
assert Solution().sumCounts([1, 1]) == 3
assert Solution().sumCounts([1, 2, 3]) == 20

## 推演与模板边界

`[1,2,1]` 的 d 依次为 `[1,0,0]`、`[2,1,0]`、`[2,2,1]`；ending 为 1、5、9，累计答案为 15。

- 区间为闭区间，模板要求非空数组且 `0 <= left <= right < n`。
- `_apply` 更新区间和时乘长度，lazy 用累加，传给孩子后清零。
- 树与 ending 保留精确整数，answer 每轮取模。
- 此模板支持区间加与求和；区间赋值、最大值需要另外设计维护规则。

建树 O(n)，每轮查询与更新各 O(log n)，总时间 O(n log n)，空间 O(n)，树递归深度 O(log n)。